# ReportLens — OCR fine-tuning (TrOCR) on Colab

Fine-tunes `microsoft/trocr-small-printed` on **synthetic lab-report lines** and reports
Character Error Rate (CER). Run the cells top-to-bottom.

**Before you start:** set the GPU runtime →  *Runtime → Change runtime type → Hardware
accelerator: **T4 GPU** → Save*.

Lower CER is better (0.0 = perfect). Aim for well under ~0.1. Training takes roughly
30–45 min on a T4 with the default settings. The engineering story here is the whole
pipeline: synthetic data → classical line segmentation → a fine-tuned recogniser, with
the same crop preprocessing used in training and inference to avoid train/serve skew.

## 1. Check the GPU is on

In [ ]:
!nvidia-smi

## 2. Clone the repo (always pulls the latest code)

Pulls in both `ocr_engine` and the `data_synthesis` package that generates the training
data. If the repo is already there from an earlier run, it is **reset to the latest
`main`** — otherwise a stale checkout would silently keep running old code.

The cell prints the commit it's running, so you can confirm you have the newest fixes.

In [ ]:
import os
%cd /content
# Robust clone: plain Python if/else with single-line shell commands (a multi-line bash
# `if` in a `!` cell is fragile in Colab). If the repo exists from an earlier run it is
# reset to the latest main so pushed fixes always take effect.
if os.path.isdir('reportlens'):
    !cd reportlens && git fetch -q origin && git reset -q --hard origin/main
else:
    !git clone -q https://github.com/Satwiksingh123/reportlens-backend-.git reportlens
%cd /content/reportlens
print('--- running commit ---')
!git log --oneline -1

## 3. Install the training dependencies (heavy — ~2-3 min)

Pins `transformers<5` (v5 changed tokenizer loading and breaks TrOCR) and installs
`sentencepiece`/`protobuf`, which TrOCR needs to build its tokenizer.

If pip prints a message about restarting the runtime, do *Runtime → Restart session*, then
re-run this cell and continue. Training itself runs in a fresh process, so it picks up the
right versions either way.

In [ ]:
!pip install -q -e "services/ocr_engine[train]"
!pip install -q pillow  # data_synthesis rendering
# TrOCR builds its fast tokenizer by converting the slow one, which needs sentencepiece
# + protobuf. Install explicitly so the conversion can't fail on a fresh Colab runtime.
!pip install -q sentencepiece protobuf

# Check versions in a SUBPROCESS. A plain `import transformers` here would report whatever
# this kernel loaded earlier in the session, not what is actually installed on disk.
!python -c "import transformers, sentencepiece; print('transformers', transformers.__version__); print('sentencepiece OK')"

## 4. Train

Generates synthetic reports, crops each labelled **word** (kept near-square, TrOCR's sweet
spot), fine-tunes TrOCR, prints the held-out CER, and saves the model to
`services/ocr_engine/artifacts/trocr-lab`.

First run downloads the base TrOCR weights (~300 MB). Roughly 25–40 min on a T4. Output is
quiet on purpose (a progress line every 100 steps) so the browser tab stays responsive.

In [ ]:
import os
# Self-bootstrap: clone the repo if a prior cell didn't (so this cell works even if run on
# its own). Then install deps and train.
%cd /content
if not os.path.isdir('reportlens'):
    !git clone -q https://github.com/Satwiksingh123/reportlens-backend-.git reportlens
%cd /content/reportlens
!pip install -q -e "services/ocr_engine[train]" pillow sentencepiece protobuf
%cd /content/reportlens/services/ocr_engine
# Word-level training on the stronger trocr-base-printed. Words give many crops per report,
# so 250 reports is plenty. batch 8 keeps the bigger model within the T4's memory.
!python -m ocr_engine.train_trocr --num-reports 250 --epochs 3 --batch-size 8 \
    --out artifacts/trocr-lab

## 5. Quick sanity check on one synthetic page

Runs the **full pipeline** (classical line segmentation + your fine-tuned recogniser) on a
freshly generated report and prints the **ground truth next to the OCR output** so you can
compare them directly.

In [ ]:
import os
# Self-bootstrap so this works even if run on its own. Runs in a subprocess so it uses the
# installed library versions, not whatever this kernel imported earlier.
%cd /content
if not os.path.isdir('reportlens'):
    !git clone -q https://github.com/Satwiksingh123/reportlens-backend-.git reportlens
%cd /content/reportlens/services/ocr_engine
!python -m ocr_engine.sanity_check --model-dir artifacts/trocr-lab

## 6. Download the fine-tuned model
Zips the model folder so you can save it (or upload it to the API host later). It is too
large for git — keep it as a release asset / on Drive, not in the repo.

In [ ]:
%cd /content/reportlens/services/ocr_engine
!cd artifacts && zip -qr trocr-lab.zip trocr-lab
from google.colab import files
files.download('artifacts/trocr-lab.zip')

### (Optional) Save to Google Drive instead
```python
from google.colab import drive; drive.mount('/content/drive')
!cp artifacts/trocr-lab.zip /content/drive/MyDrive/
```